# Funding Carry 01 — long spot / short perpetual funding harvest on Binance

Pre-registration: `discussions/Funding_Carry/001_claude_branch_preregistration.md`. Everything that decides the result is fixed in
`src/data_engineering/modeling/funding_carry.py` (`registration()` below).

- **Trade:** long $u$ spot, short $u$ USD-M perpetual ($u = 1/S_\text{entry}$), delta-neutral; the short perpetual receives funding when it is positive and pays it when negative.
- **C (primary):** enter when the trailing 7-day annualized funding $f_7 \ge 10\%$, exit when $f_7 < 3\%$; daily decisions at 00:05 UTC.
- **U (benchmark):** always in.
- **Costs:** 17 bp per side (spot 11 + perp 6); full round trip on every +40% rebalancing reset. Capital: 1.5 per unit notional (2× on the perp).
- **Verdict (C):** G1 bootstrap CI of net return on reserved capital $>0$, G2 DEV and TEST both $>0$, G3 ≥ 60% of assets net positive, G4 net on deployed capital ≥ 4%/yr.

Environment: `FC01_SYNTHETIC=1` (smoke run to a temp dir), `FC01_ASSETS` (comma list; default = the registered universe).

In [ ]:
from pathlib import Path
import json, os, sys, tempfile, time, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from dataclasses import replace
from dotenv import load_dotenv

warnings.filterwarnings('ignore', category=RuntimeWarning)
pd.set_option('display.max_columns', 40); pd.set_option('display.width', 250)
load_dotenv()
PROJECT_ROOT = Path.cwd().resolve()
while PROJECT_ROOT != PROJECT_ROOT.parent and not (PROJECT_ROOT / 'src' / 'data_engineering').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT / 'src'))
from data_engineering.modeling import funding_carry as fc

SYNTHETIC = os.getenv('FC01_SYNTHETIC', '').strip() == '1'
ASSETS = [a.strip().upper() for a in os.getenv('FC01_ASSETS', ','.join(fc.UNIVERSE)).split(',') if a.strip()]
REGISTERED_RUN = (not SYNTHETIC) and ASSETS == list(fc.UNIVERSE)
if SYNTHETIC:
    OUT = Path(tempfile.mkdtemp(prefix='fc01_'))
else:
    DATA_ROOT = Path(os.environ['DERIBIT_DATA_ROOT']).expanduser().resolve()
    BIN = Path(os.getenv('BINANCE_UM_DIR') or DATA_ROOT.parent / 'Binance').expanduser().resolve()
    OUT = PROJECT_ROOT / 'notebooks' / 'Funding_Carry' / '01'
OUT.mkdir(parents=True, exist_ok=True)
print(fc.selftest())
print('assets', len(ASSETS), '| registered run' if REGISTERED_RUN else '| NOT the registered run', '| out', OUT)
print(json.dumps(fc.registration(), indent=1))

## 1. Data: Binance spot 1m, USD-M perpetual 1m and funding prints → daily frames

In [ ]:
FRAMES, FUND, missing = {}, {}, []
if SYNTHETIC:
    S, P, F = fc.synthetic(800, seed=11, assets=('AAA', 'BBB', 'CCC', 'DDD'))
    for a in S:
        FRAMES[a] = fc.daily_frame(S[a], P[a], F[a]); FUND[a] = F[a]
else:
    for a in ASSETS:
        sym = f'{a}USDT'
        pp, sp, fp_ = BIN / f'{sym}_um_1m.parquet', BIN / 'basis' / 'spot' / f'{sym}_spot_1m.parquet', BIN / 'basis' / 'funding' / f'{sym}_funding.parquet'
        miss = [str(x.name) for x in (pp, sp, fp_) if not x.exists()]
        if miss:
            missing.append({'asset': a, 'missing': ','.join(miss)}); continue
        f = fc.load_funding(fp_)
        FRAMES[a] = fc.daily_frame(fc.load_close(sp), fc.load_close(pp), f); FUND[a] = f
print('loaded', len(FRAMES), 'assets; missing:', missing)
PROFILE = fc.funding_profile(FRAMES, FUND)
PROFILE.to_csv(OUT / '01_funding_profile.csv', index=False)
display(PROFILE.round(4))

## 2. Strategies C (primary) and U (benchmark), base costs and the low-cost sensitivity

In [ ]:
LOW = fc.Params(side_cost=(fc.LOW_SPOT_COST_BP + fc.LOW_PERP_COST_BP) / 1e4)
RUNS, SUM = {}, []
for name, lab, prm in (('C', 'C', fc.Params()), ('U', 'U', fc.Params()), ('C_lowcost', 'C', LOW), ('U_lowcost', 'U', LOW)):
    sims, port, s = fc.run_all(FRAMES, FUND, prm, lab)
    s['strategy'] = name
    RUNS[name] = (sims, port)
    SUM.append({**s, **fc.verdict(s)})
SUM = pd.DataFrame(SUM); SUM.to_csv(OUT / '01_summary.csv', index=False)
display(SUM.set_index('strategy').T)
for name in ('C', 'U'):
    RUNS[name][1].assign(strategy=name).to_csv(OUT / f'01_portfolio_daily_{name}.csv', index=False)

## 3. Per asset, episodes and the threshold grid (descriptive only — not a gate)

In [ ]:
rows = []
for name in ('C', 'U'):
    sims = RUNS[name][0]
    for a, s in sims.items():
        dep = float((fc.CAPITAL_PER_NOTIONAL * s['pos']).sum())
        rows.append({'strategy': name, 'asset': a, 'valid_days': int(FRAMES[a]['valid'].sum()), 'days_in': int(s['pos'].sum()),
                     'funding': s['funding'].sum(), 'basis': s['basis'].sum(), 'cost': s['cost'].sum(), 'net': s['net'].sum(),
                     'ann_deployed': 365 * s['net'].sum() / dep if dep else np.nan, 'resets': int(s['reset'].sum()),
                     'episodes': int(s['episode'].max() + 1)})
PER = pd.DataFrame(rows); PER.to_csv(OUT / '01_per_asset.csv', index=False)
display(PER[PER.strategy == 'C'].sort_values('ann_deployed').round(4))
EP = fc.episode_table(RUNS['C'][0]); EP.to_csv(OUT / '01_episodes_C.csv', index=False)
print('C episodes:', len(EP), '| net<0:', int((EP.net < 0).sum()), '| median days', EP.days.median())
display(EP.sort_values('worst_cum').head(10).round(4))
grid = []
for e in fc.GRID_ENTRY:
    for x in fc.GRID_EXIT:
        if x > e: continue
        _, _, s = fc.run_all(FRAMES, FUND, fc.Params(entry=e, exit=x), 'C')
        grid.append({'entry': e, 'exit': x, **{k: s[k] for k in ('ann_reserved', 'ann_deployed', 'ann_reserved_dev', 'ann_reserved_test', 'share_days_active', 'episodes')}})
GRID = pd.DataFrame(grid); GRID.to_csv(OUT / '01_grid.csv', index=False)
display(GRID.pivot(index='entry', columns='exit', values='ann_deployed').round(4))

## 4. Verdict

In [ ]:
c = SUM.set_index('strategy').loc['C']
decision = {'version': fc.VERSION, 'registered_run': REGISTERED_RUN, 'synthetic': SYNTHETIC,
            'assets_loaded': len(FRAMES), 'missing': missing,
            'C': {k: (c[k].item() if hasattr(c[k], 'item') else c[k]) for k in
                  ('G1_net_ci_positive', 'G2_dev_and_test_positive', 'G3_breadth', 'G4_beats_cash_hurdle', 'verdict',
                   'ann_reserved', 'ann_reserved_lo', 'ann_reserved_hi', 'ann_deployed', 'ann_reserved_dev', 'ann_reserved_test',
                   'max_dd_reserved', 'assets_net_positive', 'assets_scored')},
            'U_verdict': SUM.set_index('strategy').loc['U', 'verdict'],
            'C_lowcost_verdict': SUM.set_index('strategy').loc['C_lowcost', 'verdict']}
decision['branch_verdict'] = decision['C']['verdict']
(OUT / '01_decision.json').write_text(json.dumps(decision, indent=1, default=str))
print(json.dumps(decision, indent=1, default=str))

## 5. Equity on reserved capital and the cross-asset funding level

In [ ]:
fig, ax = plt.subplots(2, 1, figsize=(12, 7), sharex=True)
for name in ('C', 'U'):
    p = RUNS[name][1]
    ax[0].plot(pd.to_datetime(p['b']), 100 * p['ret_reserved'].cumsum(), label=f'{name} net (% of reserved capital)')
ax[0].axvline(fc.DEV_END, c='k', ls=':', lw=1); ax[0].axhline(0, c='k', lw=.5); ax[0].legend(); ax[0].set_title('Cumulative net return, not compounded')
f7 = pd.concat({a: d.set_index('b')['f7'] for a, d in FRAMES.items()}, axis=1)
ax[1].plot(f7.index, 100 * f7.median(axis=1), label='median f7 across assets')
ax[1].fill_between(f7.index, 100 * f7.quantile(.25, axis=1), 100 * f7.quantile(.75, axis=1), alpha=.3, label='IQR')
ax[1].axhline(100 * fc.ENTRY, c='g', ls='--', lw=1); ax[1].axhline(100 * fc.EXIT, c='r', ls='--', lw=1); ax[1].axhline(0, c='k', lw=.5)
ax[1].set_ylabel('% annualized'); ax[1].legend()
plt.tight_layout(); plt.savefig(OUT / '01_equity_funding.png', dpi=110); plt.show()